In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

os.chdir('../cross-rag')

def parse_result_file(file_path):
    results_dict = {}  # Use a dictionary to keep only the last result per dataset
    with open(file_path, 'r') as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if not line:
            i += 1
            continue
        if '_zeroshot_' in line:
            dataset = line.split('_zeroshot_')[0]
            if i + 1 < len(lines):
                metric_line = lines[i + 1].strip()
                mse_match = re.search(r'mse:([\d.]+)', metric_line)
                mae_match = re.search(r'mae:([\d.]+)', metric_line)
                if mse_match and mae_match:
                    # Update dictionary - later results will overwrite earlier ones
                    results_dict[dataset] = {
                        'dataset': dataset,
                        'mse': float(mse_match.group(1)),
                        'mae': float(mae_match.group(1))
                    }
                i += 2
            else:
                i += 1
        else:
            i += 1
    return pd.DataFrame(list(results_dict.values()))

In [19]:
PATH = f'results/forecast_evaluation'
settings = os.listdir(PATH)
settings = [x for x in settings if 'dualhead' in x]
settings = [x for x in settings if 'dropout' not in x]
settings = [x for x in settings if 'random' not in x]
settings = [x for x in settings if 'alpha' not in x]
settings = [x for x in settings if 'R_as_key' in x]
settings = [x for x in settings if 'cosine' in x]
settings = [x for x in settings if 'moe' not in x]
settings = [x for x in settings if 'ablation' not in x]
settings = sorted(settings)
len(settings)
os.chdir(PATH)

In [22]:
settings_learnable = [x for x in settings if 'learn' in x]
settings_nonlearnable = [x for x in settings if 'learn' not in x]

In [35]:
for x in settings_learnable:
    print(x)
    df = parse_result_file(x)
    print(df['mse'].mean().round(3))

zeroshot_Chronos_lb512_X-cosine-norm_k10_TabPFN_dualhead_learnable_R_as_key.txt
0.196
zeroshot_Chronos_lb512_X-cosine-norm_k15_TabPFN_dualhead_learnable_R_as_key.txt
0.196
zeroshot_Chronos_lb512_X-cosine-norm_k1_TabPFN_dualhead_learnable_R_as_key.txt
0.198
zeroshot_Chronos_lb512_X-cosine-norm_k20_TabPFN_dualhead_learnable_R_as_key.txt
0.197
zeroshot_Chronos_lb512_X-cosine-norm_k3_TabPFN_dualhead_learnable_R_as_key.txt
0.197
zeroshot_Chronos_lb512_X-cosine-norm_k5_TabPFN_dualhead_learnable_R_as_key.txt
0.197


In [27]:
df['mse'].mean()

0.19317142857142855

In [28]:
lamb3=[]
lamb4=[]
lamb5=[]
lamb6=[]
lamb7=[]
lamb8=[]
for x in settings_nonlearnable:
    df = parse_result_file(x)
    mse = df['mse'].mean()
    if 'l0.3' in x:
        lamb3.append(mse)
    if 'l0.4' in x:
        lamb4.append(mse)
    if 'l0.5' in x:
        lamb5.append(mse)
    if 'l0.6' in x:
        lamb6.append(mse)
    if 'l0.7' in x:
        lamb7.append(mse)        
    if 'l0.8' in x:
        lamb8.append(mse)        

In [31]:
print(np.min(lamb3))
print(np.min(lamb4))
print(np.min(lamb5))
print(np.min(lamb6))
print(np.min(lamb7))
print(np.min(lamb8))

0.19931428571428572
0.19577142857142857
0.1943285714285714
0.1928857142857143
0.19242857142857142
0.19407142857142853


In [ ]:
# learnable 4 5 6 7 8 
# 0.196 0.196 0.194 0.193 0.192 0.194